# Exercise 14: Amplitude-dependent tune

**Block 4: Resonances**

**Learning objectives**
* Measure the tune as a function of the action, $Q(J)$, for a ring with an octupole.
* Compare with the first-order formula $\Delta Q = k_3L \beta^2 J / 16\pi$.
* Understand what happens near a resonance when $Q$ depends on $J$: particles lock onto the resonance
  (islands), and growth stops because a particle "walks out" of the resonance condition.

**Motivation.** In a linear ring all particles have the same tune. An octupole makes particles with larger
amplitude oscillate faster (or slower). This **amplitude detuning** is behind the islands of Exercise 11,
and it is exactly the tune *spread* that Landau damping needs (Block 5).

> **Student version.** Fill in the cells marked `# YOUR CODE HERE` and run the check cells after them: they tell you when you are right.

In [ ]:
# --- setup: make `beamlib` importable --------------------------------------
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "beamlib").is_dir())
sys.path.insert(0, str(_root))

import numpy as np
import matplotlib.pyplot as plt
from beamlib import lattices as L, tracking as T, diagnostics as D, plotting as P
from beamlib.plotting import COLORS, MM
from beamlib.widgets import interactive_plot
P.setup()
BETA = 4.0

## 1. The first-order formula

A thin octupole gives $\Delta p_x = −(k_3L/6) x^3$ (for $y$ = 0). Compare with a thin quadrupole,
$\Delta p_x = −kl x$, which shifts the tune by $\Delta Q = \beta kl / 4\pi$ (lectures). The octupole is like a
quadrupole whose strength grows with $x^2$: $kl_\text{eff} = (k_3L/6) x^2$.

Averaging over one oscillation ($x = \sqrt{2J\beta} \cos\phi$, and the average of $\cos^4\phi$ is 3/8) gives

$$\Delta Q(J) = \frac{k_3L\,\beta^2}{16\pi}\,J .$$

The tune grows linearly with the action $J$ (for $k_3L > 0$).

**Task.** Implement the formula.

In [ ]:
def detuning_theory(J, k3l: float, beta: float):
    """First-order tune shift of a thin octupole: k3l beta^2 J / (16 pi)."""
    # YOUR CODE HERE
    raise NotImplementedError("Replace this line with your code")

## 2. Measuring $Q(J)$

**Task.** Write `tune_vs_action(ring, amplitudes, n_turns)`: start one particle per amplitude at
$\hat{x} = a$, $\hat{p} = 0$, $\hat{y} = 0$; track; return for each particle its average action $J$ and its tune
(`D.tune_naff` of the complex signal $\hat{x} − i \hat{p}$).

In [ ]:
def tune_vs_action(ring, amplitudes, n_turns: int = 1024, angle: float = 0.0):
    """Average action J [m] and tune Q of particles started at x_hat = a cos(angle), p_hat = -a sin(angle)."""
    a = np.asarray(amplitudes, dtype=float)
    # YOUR CODE HERE
    raise NotImplementedError("Replace this line with your code")

**Predict first.** With $Q_0$ = 0.21 and $k_3L = 1000 \text{ m}^{-3}$, what is the tune of a particle with
$\hat{x} = 10 \sqrt{\mu\text{m}}$ ($J = \hat{x}^2/2 = 50 \mu\text{m}$)? Use the formula.

*Your prediction:*

In [ ]:
Q0, k3l = 0.21, 1000.0
ring = L.HenonRing(Q0, 0.30, BETA, 0.0, BETA, 0.0, knl=[0, 0, 0, k3l])
amps = np.linspace(0.2e-3, 12e-3, 40)
J, Q = tune_vs_action(ring, amps)
dQ_th = detuning_theory(J, k3l, BETA)

small = J < 5e-6
assert np.allclose(Q[small] - Q0, dQ_th[small], rtol=0.03), "small amplitudes should follow the formula"
print(f"at J = {J[-1] * 1e6:.0f} mm mrad: measured dQ = {Q[-1] - Q0:.4f}, first-order formula {dQ_th[-1]:.4f}")
print("GOOD! The first-order detuning formula works at small amplitude")

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(J * 1e6, Q, "o", color=COLORS["x"], label="tracking (NAFF)")
ax.plot(J * 1e6, Q0 + dQ_th, "k--", label=r"$Q_0 + k_3L\beta^2 J/16\pi$")
ax.set_xlabel(r"$J$ [$\mu$m]"); ax.set_ylabel(r"$Q_x$"); ax.legend()
ax.set_title(rf"Octupole, $Q_0$ = {Q0}, $k_3L$ = {k3l:.0f} $1/\text{{m}}^3$")
plt.show()

At small $J$ the agreement is excellent. At large $J$ the measured tune bends away from the straight line:
the formula is only the *first-order* term (the next terms are $\propto (k_3L)^3 J^2$).

## 3. Close to a resonance

**Predict first.** Now $Q_0 = 0.24$, just below 1/4. The tune increases with $J$. What happens to $Q(J)$
when it reaches 0.25?

*Your prediction:*

In [ ]:
ring_r = L.HenonRing(0.24, 0.30, BETA, 0.0, BETA, 0.0, knl=[0, 0, 0, 1000.0])
amps = np.linspace(0.2e-3, 14e-3, 120)
J_axis, Q_axis = tune_vs_action(ring_r, amps, angle=0.0)
J_diag, Q_diag = tune_vs_action(ring_r, amps, angle=np.pi / 4)
locked = np.abs(Q_diag - 0.25) < 1e-6
print(f"{locked.sum()} particles started on the diagonal have Q = 0.25 exactly (to 1e-6): they are in the islands")
assert locked.sum() >= 5

fig, axs = plt.subplots(1, 2, figsize=(13, 5))
axs[0].plot(amps * MM, Q_axis, "o", ms=3, color=COLORS["x"], label=r"started on the $\hat{x}$ axis")
axs[0].plot(amps * MM, Q_diag, "s", ms=3, color=COLORS["accent"], label="started on the diagonal")
axs[0].axhline(0.25, color="k", ls=":", lw=1)
axs[0].set_xlabel(r"starting amplitude [$\sqrt{\mu\text{m}}$]"); axs[0].set_ylabel(r"$Q_x$"); axs[0].legend()
axs[0].set_title(r"$Q$ stops at 1/4 inside the islands")
for a, ang, col in [(amps, 0.0, COLORS["x"]), (amps, np.pi / 4, COLORS["accent"])]:
    bn = np.zeros((4, len(a[::6]))); bn[0, :] = a[::6] * np.cos(ang); bn[1, :] = -a[::6] * np.sin(ang)
    zn = ring_r.normalise(T.track(ring_r.denormalise(bn), ring_r.one_turn_map, 600).coords)
    axs[1].scatter(zn[:, 0, :] * MM, zn[:, 1, :] * MM, s=0.2, color=col)
axs[1].set_aspect("equal"); P.phase_space_labels(axs[1], "x", True); axs[1].set_xlim(-17, 17); axs[1].set_ylim(-17, 17)
axs[1].set_title("the same particles in phase space")
plt.show()

Particles inside the four islands all have *exactly* $Q = 1/4$: they circle around an island centre while
hopping from island to island, so on average they advance by exactly a quarter turn per turn. They are
**locked** to the resonance. Particles that start on the $\hat{x}$ axis go through the unstable fixed points
instead and their tune jumps across 1/4.

Why does the particle not run away like at the half-integer stopband? As soon as it is pushed to a larger
amplitude its tune changes, so it is no longer *on* the resonance, and the kicks stop adding up. The
detuning **limits the growth**. Section 4 shows this directly.

**Explore.** Change $Q_0$ and $k_3L$. Where do the islands appear? What happens with $k_3L$ < 0?

In [ ]:
def show_detuning(Q0, k3l):
    rg = L.HenonRing(Q0, 0.30, BETA, 0.0, BETA, 0.0, knl=[0, 0, 0, k3l])
    a = np.linspace(0.2e-3, 14e-3, 50)
    J, Q = tune_vs_action(rg, a, n_turns=512, angle=np.pi / 4)
    fig, axs = plt.subplots(1, 2, figsize=(12, 4.5))
    axs[0].plot(J * 1e6, Q, "o", ms=3, color=COLORS["x"], label="tracking")
    axs[0].plot(J * 1e6, Q0 + detuning_theory(J, k3l, BETA), "k--", label="first order")
    for q in (0.2, 0.25, 1 / 3):
        axs[0].axhline(q, color="0.6", ls=":", lw=1)
    axs[0].set_xlabel("J [mm mrad]"); axs[0].set_ylabel("Qx"); axs[0].legend()
    bn = np.zeros((len(a[::3]), 4)); bn[:, 0] = a[::3] * np.cos(np.pi / 4); bn[:, 1] = -a[::3] * np.sin(np.pi / 4)
    zn = rg.normalise(T.track(rg.denormalise(bn), rg.one_turn_map, 400).coords)
    axs[1].scatter(zn[..., 0] * MM, zn[..., 1] * MM, s=0.2, c=np.broadcast_to(np.arange(zn.shape[1]), zn.shape[:2]))
    axs[1].set_aspect("equal"); axs[1].set_xlim(-17, 17); axs[1].set_ylim(-17, 17)
    P.phase_space_labels(axs[1], "x", True)
    axs[0].set_title(f"Q0 = {Q0:.3f}, k3L = {k3l:.0f} 1/m³")
    plt.show()

interactive_plot(show_detuning, Q0=(0.15, 0.30, 0.002, 0.24), k3l=(-2000.0, 2000.0, 100.0, 1000.0))

## 4. Detuning can stop resonance growth

Back to the half-integer stopband of Exercise 11: $Q_0 = 0.49$ with a quadrupole error $k_1L = 0.05 \text{m}^{-1}$.
The stopband is 0.468 < $Q$ < 0.5, so without an octupole every particle grows exponentially.
Now add an octupole, with either sign.

**Predict first.** Which sign of $k_3L$ moves the tune *out* of the stopband as the amplitude grows?
Will the particle still grow forever?

*Your prediction:*

In [ ]:
J0 = 1e-8
fig, ax = plt.subplots(figsize=(9, 4.5))
J_curves = {}
for k3, col, lab in [(0.0, COLORS["y"], "quad error only"),
                     (+1000.0, COLORS["accent"], r"quad error + octupole, $k_3L = +1000$"),
                     (-1000.0, COLORS["x"], r"quad error + octupole, $k_3L = -1000$")]:
    rg = L.HenonRing(0.49, 0.30, BETA, 0.0, BETA, 0.0, knl=[0, 0.05, 0, k3])
    res = T.track(rg.denormalise(np.array([[np.sqrt(2 * J0)], [0], [0], [0]])), rg.one_turn_map, 1000)
    zn = rg.normalise(res.coords)[:, :, 0]          # one particle: (turns + 1, 4)
    J_curves[k3] = 0.5 * (zn[:, 0]**2 + zn[:, 1]**2)
    ax.semilogy(J_curves[k3] * 1e6, color=col, label=lab + ("" if res.survived[0] else "  (lost)"))
ax.set_xlabel("turn"); ax.set_ylabel(r"$J$ [$\mu$m]"); ax.legend(fontsize=8)
ax.set_title(r"$Q_x = 0.49$, $k_1L = 0.05 1/\text{m}$: inside the stopband 0.468 < $Q$ < 0.5")
plt.show()

Jmax = np.nanmax(J_curves[-1000.0])
assert np.all(np.isfinite(J_curves[-1000.0])) and Jmax < 1e-4, "with k3L < 0 the growth saturates"
assert np.isnan(J_curves[1000.0][-1]), "with k3L > 0 the particle is lost"
print(f"k3L = -1000: growth stops at J = {Jmax * 1e6:.0f} mm mrad, where the first-order detuning is "
      f"{detuning_theory(Jmax, -1000.0, BETA):.3f},\ncomparable to the 0.022 needed to leave the stopband "
      f"(0.49 -> 0.468; the exact shift is a bit larger than first order, see section 2)")

With $k_3L < 0$ the action first grows exponentially (the particle is in the stopband), then stops and
oscillates: the tune has moved *down*, out of the stopband. The particle "walked out" of the resonance
condition. This is also why the islands of a nonlinear resonance are bounded, while a *linear* resonance
(no detuning) is not.

With $k_3L > 0$ the particle is lost, even though its tune is pushed up towards the edge at 0.5. The
reason: an octupole does not only detune, it also acts like an extra quadrupole error $\propto k_3L x^2$ that
itself drives the half-integer resonance. With the same sign as $k_1L$ it *adds* to the error and the
stopband widens as the particle grows. The sign matters!

## Reflection questions

1. The detuning formula has $\beta^2$. Where in a ring would you put an octupole to get a large tune spread
   with a weak magnet?
2. A beam with emittance $\varepsilon_\text{rms}$ has ⟨$J$⟩ = $\varepsilon_\text{rms}$ (Exercise 6). Estimate the rms tune
   spread of a 1 mm mrad beam due to our octupole ($k_3L = 1000 \text{m}^{-3}$, $\beta$ = 4 m).
3. Is amplitude detuning good or bad? Give one argument for each.